# 04 - Modeling
Train a baseline clustering model.

In [ ]:
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import os

# Tạo thư mục app nếu chưa có để lưu mô hình
os.makedirs('../app', exist_ok=True)

print("🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH...")

# ==========================================
# 1. TRAIN MÔ HÌNH K-MEANS (PHÂN CỤM KHÁCH HÀNG)
# ==========================================
print("\n--- 1. Đang huấn luyện K-Means (RFM) ---")
df_rfm = pd.read_csv('../data/processed/ml_features_rfm.csv')

# Chỉ lấy 3 cột R, F, M để train
X_rfm = df_rfm[['recency', 'frequency', 'monetary']]

# Chuẩn hóa dữ liệu (quan trọng cho K-Means)
scaler = StandardScaler()
X_rfm_scaled = scaler.fit_transform(X_rfm)

# Huấn luyện chia làm 4 cụm khách hàng
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X_rfm_scaled)

# Lưu lại mô hình và bộ chuẩn hóa
joblib.dump({'model': kmeans, 'scaler': scaler}, '../app/kmeans_model.pkl')
print("✅ Đã lưu não bộ K-Means vào: app/kmeans_model.pkl")


# ==========================================
# 2. TRAIN MÔ HÌNH XGBOOST (DỰ BÁO HOÀN TRẢ)
# ==========================================
print("\n--- 2. Đang huấn luyện XGBoost (Dự báo hoàn trả) ---")
df_xgb = pd.read_csv('../data/processed/ml_features_xgboost.csv')

# Tách biến mục tiêu (y) và dữ liệu đầu vào (X)
y = df_xgb['is_returned']
X = df_xgb.drop(columns=['is_returned'])

# Chia tập Train/Test theo tỷ lệ nhãn để giữ cân bằng
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Tính scale_pos_weight cho dữ liệu mất cân bằng
imbalance_ratio = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))

base_model = xgb.XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=imbalance_ratio,
    eval_metric='logloss'
)

param_distributions = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 1, 3],
    'reg_alpha': [0, 0.5, 1],
    'reg_lambda': [1, 2, 5],
}

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=25,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
    verbose=1,
    n_jobs=-1,
    refit=True,
    random_state=42,
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("\n✅ Tìm được tham số tốt nhất cho XGBoost:")
print(search.best_params_)
print(f"ROC AUC trung bình trong CV: {search.best_score_:.4f}")

# Đánh giá trên tập kiểm định
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
print("\nBảng điểm (Report) của XGBoost trên tập Test:")
print(classification_report(y_test, y_pred, digits=4))
print(f"ROC AUC (Test): {roc_auc_score(y_test, y_prob):.4f}")

# Lưu mô hình tốt nhất
joblib.dump(best_model, '../app/xgboost_returns_model.pkl')
print("✅ Đã lưu não bộ XGBoost vào: app/xgboost_returns_model.pkl")

# Lấy ra Top 10 yếu tố quan trọng nhất gây ra việc trả hàng
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10)

print("\n💡 TOP 10 YẾU TỐ QUYẾT ĐỊNH VIỆC KHÁCH TRẢ HÀNG LÀ:")
print(importance.to_string(index=False))

🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH...

--- 1. Đang huấn luyện K-Means (RFM) ---
✅ Đã lưu não bộ K-Means vào: app/kmeans_model.pkl

--- 2. Đang huấn luyện XGBoost (Dự báo hoàn trả) ---

Bảng điểm (Report) của XGBoost trên tập Test:
              precision    recall  f1-score   support

           0       0.35      0.92      0.51      1130
           1       0.99      0.81      0.89     10080

    accuracy                           0.82     11210
   macro avg       0.67      0.86      0.70     11210
weighted avg       0.92      0.82      0.85     11210

✅ Đã lưu não bộ XGBoost vào: app/xgboost_returns_model.pkl

💡 TOP 5 YẾU TỐ QUYẾT ĐỊNH VIỆC KHÁCH TRẢ HÀNG LÀ:
              Feature  Importance
          productcost    0.599878
productsubcategorykey    0.114463
        profit_margin    0.076632
         productprice    0.042113
         annualincome    0.024941


In [2]:
import joblib
import numpy as np

# 1. "Rã đông" file K-Means
kmeans_data = joblib.load('../app/kmeans_model.pkl')
kmeans_model = kmeans_data['model']
scaler = kmeans_data['scaler']

# 2. Mở một file văn bản mới để ghi thông tin vào
with open('../app/kmeans_readable_info.txt', 'w', encoding='utf-8') as f:
    f.write("=== HỒ SƠ HUẤN LUYỆN K-MEANS ===\n\n")
    
    # Ghi thông số thuật toán
    f.write(f"1. Số cụm khách hàng (n_clusters): {kmeans_model.n_clusters}\n")
    
    # Ghi tọa độ của 4 cụm (Centroids)
    f.write("\n2. Tọa độ của các tâm cụm (Centroids):\n")
    f.write(np.array2string(kmeans_model.cluster_centers_, separator=', '))
    
    # Ghi thông số chuẩn hóa
    f.write("\n\n3. Trạng thái bộ chuẩn hóa (Scaler):\n")
    f.write(f"- Trung bình (Mean) của các biến gốc: {scaler.mean_}\n")
    f.write(f"- Độ lệch (Scale) của các biến gốc: {scaler.scale_}\n")

print("✅ Đã xuất thông tin thành file TEXT! Bạn hãy mở file app/kmeans_readable_info.txt để xem.")

✅ Đã xuất thông tin thành file TEXT! Bạn hãy mở file app/kmeans_readable_info.txt để xem.


In [ ]:
import joblib
import pandas as pd

# 1. Load trực tiếp mô hình XGBoost
xgb_model = joblib.load('../app/xgboost_returns_model.pkl')

# Đọc lại data để biết tên biến đầu vào
try:
    df_xgb = pd.read_csv('../data/processed/ml_features_xgboost.csv')
    feature_names = [c for c in df_xgb.columns if c != 'is_returned']
except FileNotFoundError:
    feature_names = None

with open('../app/xgboost_readable_info.txt', 'w', encoding='utf-8') as f:
    f.write("=== HỒ SƠ HUẤN LUYỆN XGBOOST ===\n\n")
    
    f.write("1. Cấu hình tham số (Hyperparameters):\n")
    params = xgb_model.get_params()
    for key, value in params.items():
        if value is not None:
            f.write(f" - {key}: {value}\n")
            
    f.write("\n2. Mức độ tác động của các yếu tố (Feature Importances):\n")
    importances = xgb_model.feature_importances_
    if feature_names is not None and len(feature_names) == len(importances):
        for name, imp in zip(feature_names, importances):
            f.write(f" - {name}: {imp}\n")
    else:
        for i, imp in enumerate(importances):
            f.write(f" - Cột số {i}: {imp}\n")

print("✅ Đã xuất thông tin XGBoost thành file TEXT!")

✅ Đã xuất thông tin XGBoost thành file TEXT!
